# 07-Token Streaming & Server-Sent Events (SSE)

In Lesson 06, we solved the Concurrency Trap. By offloading our heavy PyTorch mathematics to a background thread pool, we mathematically guaranteed that the FastAPI Event Loop would never freeze.

But we are still suffering from **Generation Latency**. If an LLM takes 10 seconds to generate a 500-word response, the API waits 10 full seconds before returning one massive JSON payload. To the end-user, the screen sits completely blank. They will assume your app has crashed.

To replicate the fluid, real-time "typing" effect of ChatGPT, we must dismantle the standard HTTP Request/Response cycle and master the physics of **Server-Sent Events (SSE)**.

Let's set up our PyTorch and FastAPI environment to engineer a real-time token stream.

In [1]:
import torch
import asyncio
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread
import time
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch, FastAPI, and Streaming Environment Ready.")

✅ PyTorch, FastAPI, and Streaming Environment Ready.


# 1. The Physics of HTTP Chunking

Standard REST APIs use a rigid protocol:

1. Client sends a request.
2. Server closes the door, computes the *entire* payload, calculates the exact `Content-Length` in bytes.
3. Server opens the door and drops the massive payload on the client.

If you don't know the `Content-Length` because the AI is still generating words, you cannot use this protocol.

Instead, we use **Chunked Transfer Encoding** via **Server-Sent Events (SSE)**.
We change the HTTP response header to `Content-Type: text/event-stream`.
This tells the client's browser: *"Keep the TCP connection open indefinitely. I am going to send you fragmented packets of text. Every time you see a double newline (`\n\n`), that marks the end of a chunk. Render it immediately."*

# 2. Python Generators (The `yield` Calculus)

To feed the open HTTP connection, we cannot use a `return` statement. A `return` mathematically destroys the function's local variables and closes the stack frame.

Instead, we use **Generators** via the `yield` keyword.
When Python hits `yield`, it physically pauses the function's execution, pushes the data through the open HTTP socket, and keeps all variables safely frozen in memory until the next token is ready.

# 3. The PyTorch C++ Bottleneck

Here is the ultimate architectural challenge: **PyTorch's `model.generate()` function is a monolithic, blocking C++ loop.** You cannot simply insert a `yield` statement inside the pre-compiled C++ source code of the GPU driver.

To bridge the gap between synchronous PyTorch generation and asynchronous FastAPI yielding, Hugging Face engineered the `TextIteratorStreamer`.

### The Dual-Thread Architecture

1. **Thread 1 (The Worker)**: We take the heavy `model.generate()` function and push it onto a background CPU thread (just like we learned in Lesson 06!).
2. **The Queue**: As the GPU calculates a token, the C++ backend drops the token into a thread-safe Python `Queue`.
3. **Thread 2 (The Event Loop)**: The main FastAPI asynchronous route monitors that `Queue`. Whenever a token appears, it instantly `yield`s it over the network.


# 4. Architecting the Streaming API

Let's build a fully functional, production-ready Streaming Endpoint. We will initialize the LLM, construct the dual-thread architecture, and format the output according to the strict Server-Sent Events protocol.

In [2]:
# --- ⚙️ main.py ---

app = FastAPI(title="Enterprise LLM Streaming API")

print("--- 📥 Initializing Heavy Weights (Simulated) ---")
# In production, this goes in the @asynccontextmanager lifespan block!
# We use a tiny model like GPT-2 to safely simulate this in the notebook environment.
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
print("Model safely loaded in memory.")

# 1. Architecting the Asynchronous Generator
async def generate_token_stream(prompt: str):
    # A. Initialize the Thread-Safe Streamer
    # This acts as the mathematical bridge between the GPU and the Network
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    # B. Tokenize the input
    inputs = tokenizer([prompt], return_tensors="pt")
    
    # C. Configure generation parameters
    generation_kwargs = dict(
        inputs,
        streamer=streamer,
        max_new_tokens=20,
        temperature=0.7,
        do_sample=True
    )
    
    # D. Start the Heavy Generation in a Background Thread!
    # This prevents the blocking C++ loop from freezing our FastAPI Event Loop
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # E. The Yield Loop (Feeding the Network)
    # We iterate over the streamer queue. When a token is calculated, it appears here.
    for new_text in streamer:
        # SSE Protocol dictates we format the string as: data: <content>\n\n
        yield f"data: {new_text}\n\n"
        
        # We add a tiny async sleep to explicitly yield control back to the ASGI Event Loop,
        # ensuring absolute concurrency for other users.
        await asyncio.sleep(0.01)
        
    # Send a termination signal to the client
    yield "data: [DONE]\n\n"

# 2. Define the HTTP Route
@app.get("/v1/chat/stream")
async def chat_stream(prompt: str):
    """
    Returns a StreamingResponse instead of a standard JSON dict.
    The Content-Type is mathematically locked to 'text/event-stream'.
    """
    return StreamingResponse(
        generate_token_stream(prompt), 
        media_type="text/event-stream"
    )

print("Success! Streaming architecture compiled.")

--- 📥 Initializing Heavy Weights (Simulated) ---


2026-06-16 15:15:11.619420: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model safely loaded in memory.
Success! Streaming architecture compiled.


# 5. Simulating the Client-Side Reception

If a standard machine hits this endpoint, it doesn't receive a clean dictionary. It receives a continuous, fragmented stream of raw bytes. Let's write a Python client script to simulate exactly what the frontend (or iOS app) receives as the TCP packets arrive.

In [5]:
# --- 📡 Client Simulation Script ---

print("\n--- 🌐 Initiating Server-Sent Events Connection ---")
test_prompt = "Deep learning is fundamentally transforming"

async def simulate_client_reception():
    print(f"User Prompt: '{test_prompt}'\n")
    print("--- 📥 Intercepting Raw TCP Packets ---")
    
    # We directly execute our generator function to simulate network reception
    stream_generator = generate_token_stream(test_prompt)
    
    final_reconstructed_text = ""
    
    async for raw_packet in stream_generator:
        # Print the literal string format arriving over the wire
        print(f"Received Packet -> {repr(raw_packet)}")
        
        # The Client (JavaScript/Python) must parse the SSE protocol
        if raw_packet.startswith("data: "):
            payload = raw_packet.replace("data: ", "").replace("\n\n", "")
            if payload == "[DONE]":
                break
            
            final_reconstructed_text += payload
            
    print(f"\n✅ Final Reconstructed UI String: '{final_reconstructed_text}'")

# Run the simulation
# asyncio.run(simulate_client_reception())
await simulate_client_reception()
print("\n--- 💡 Architecture Insight ---")
print("Look at the Raw Packets. Each token arrives packaged in the rigid 'data: ... \\n\\n' format. If you open a standard Web Browser and navigate to this API URL, the browser natively understands this MIME type. It will physically print the words onto the screen one by one as they arrive from your GPU, perfectly replicating the ChatGPT user experience!")



--- 🌐 Initiating Server-Sent Events Connection ---
User Prompt: 'Deep learning is fundamentally transforming'

--- 📥 Intercepting Raw TCP Packets ---


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Received Packet -> 'data:  \n\n'
Received Packet -> 'data: \n\n'
Received Packet -> 'data: medicine, \n\n'
Received Packet -> 'data: as \n\n'
Received Packet -> 'data: evidenced \n\n'
Received Packet -> 'data: by \n\n'
Received Packet -> 'data: the \n\n'
Received Packet -> 'data: widespread \n\n'
Received Packet -> 'data: use \n\n'
Received Packet -> 'data: of \n\n'
Received Packet -> 'data: \n\n'
Received Packet -> 'data: \n\n'
Received Packet -> 'data: \n\n'
Received Packet -> 'data: cognitive-behavioral \n\n'
Received Packet -> 'data: therapies \n\n'
Received Packet -> 'data: to \n\n'
Received Packet -> 'data: treat \n\n'
Received Packet -> 'data: \n\n'
Received Packet -> 'data: obesity, \n\n'
Received Packet -> 'data: diabetes \n\n'
Received Packet -> 'data: and\n\n'
Received Packet -> 'data: [DONE]\n\n'

✅ Final Reconstructed UI String: ' medicine, as evidenced by the widespread use of cognitive-behavioral therapies to treat obesity, diabetes and'

--- 💡 Architecture Insight ---
L

## Real-World Use Case or Analogy:

Think of the difference between Standard REST and Server-Sent Events like **Publishing a Book vs. Performing a Live Play**:

* **Standard JSON API (Publishing a Book)**: An author (The GPU) writes an entire 500-page book. You have to wait 2 years for them to finish it, edit it, and print it. Finally, the publisher binds it all together (The `Content-Length` header) and drops a massive 3-pound brick of paper on your doorstep (The HTTP Response).
* **Token Streaming / SSE (The Live Play)**: You sit in a theater. The actor (The GPU) speaks their lines live. You don't have to wait 2 hours for the play to finish to understand what is happening. You process the information word-by-word, in real-time, exactly as it is generated. The connection remains constantly open, delivering packets of audio straight to your ears until the curtain falls (`[DONE]`).